In [2]:
from transformers import pipeline

In [3]:
ko_zero_shot_clf = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7', device=0)

config.json: 0.00B [00:00, ?B/s]

c:\Users\skybl\miniconda3\envs\nlp_practice\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\skybl\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-xnli-multilingual-nli-2mil7. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
import random
from transformers import pipeline

# clf모델 생성
ko_zero_shot_clf = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7', device=0)

# 라벨 및 질문지 생성
mbti_labels, personality_labels, questions = set_up()

# 메인 알고리즘
selected_questions = random.sample(questions, 3)

print('간이 MBTae 분석을 시작합니다.\n만약 나라면 어떤 생각을 할까에 대하여 답변해주세요.\n(최소 3문장 이상의 답변으로 부탁드립니다!)')

total_score = {'E': 0, 'I': 0, 'S': 0, 'N': 0, 'T': 0, 'F': 0, 'J': 0, 'P': 0}

for i, item in enumerate(selected_questions):
    print(f'\nQ{i+1}: {item['q']}')
    answer = input('답변: ')
    print(f'A{i+1}: {answer}')
    
    # 답변 분석
    for target in item['targets']:
        candidate_labels = personality_labels[target]
        output = ko_zero_shot_clf(
            answer,
            candidate_labels=candidate_labels,
            hypothesis_template='이 사람은 {}하는 사람이다.'
        )        
        for label, score in zip(output['labels'], output['scores']):
            short_label = mbti_labels[label]
            total_score[short_label] += score

# 결과 출력
result = ''
result += 'E' if total_score['E'] > total_score['I'] else 'I'
result += 'S' if total_score['S'] > total_score['N'] else 'N'
result += 'T' if total_score['T'] > total_score['F'] else 'F'
result += 'J' if total_score['J'] > total_score['P'] else 'P'
print(f'\n당신의 MBTI는 {result}입니다.')

mbti_detail = {
    'E': {'score': 0, 'description': '외향형'},
    'I': {'score': 0, 'description': '내향형'},
    'S': {'score': 0, 'description': '감각형'},
    'N': {'score': 0, 'description': '직관형'},
    'T': {'score': 0, 'description': '사고형'},
    'F': {'score': 0, 'description': '감정형'},
    'J': {'score': 0, 'description': '판단형'},
    'P': {'score': 0, 'description': '인지형'},                            
}

sections = [('E', 'I'), ('S', 'N'), ('T', 'F'), ('J', 'P')]

for former, latter in sections:
    mbti_detail[former]['score'] = round(total_score[former] / (total_score[former] + total_score[latter]), 2) * 100
    mbti_detail[latter]['score'] = 100 - mbti_detail[former]['score']
    if mbti_detail[former]['score'] > mbti_detail[latter]['score']:
        print(f'{former}({mbti_detail[former]['description']}): {mbti_detail[former]['score']}%')
    else:
        print(f'{latter}({mbti_detail[latter]['description']}): {mbti_detail[latter]['score']}%')


Device set to use cuda:0


간이 MBTae 분석을 시작합니다.
만약 나라면 어떤 생각을 할까에 대하여 답변해주세요.
(최소 3문장 이상의 답변으로 부탁드립니다!)


Q1: 북적이는 스포츠 박람회에서 최신 장비를 직접 체험하며 돌아다니는 쪽인가요, 조용한 부스에서 기술 원리를 깊이 살피는 쪽인가요?
A1: 조용한 부스에서 장비의 기술적 원리와 성능 수치를 꼼꼼히 살피는 쪽을 선호합니다. 단순히 체험하는 것보다 이 장비가 이전 모델에 비해 얼마나 효율성이 개선되었는지 데이터 시트를 확인하는 것이 더 유익하기 때문입니다. 정해진 순서대로 모든 부스를 꼼꼼히 훑어볼 것입니다.

Q2: 당신이 생각하는 이상적인 친구 관계란 무엇인가요? 활발하게 활동을 공유하는 관계인가요, 깊은 철학적 대화와 공감이 오가는 관계인가요?
A2: 서로의 사생활을 존중하며 꼭 필요한 순간에 확실한 도움을 줄 수 있는 관계가 이상적입니다. 매일 만나 활발하게 노는 것보다, 각자의 위치에서 본분에 충실하다가 가끔 만나 실질적인 조언과 정보를 교환하며 신뢰를 쌓아가는 안정적인 관계를 추구합니다.

Q3: 자율 주행 자동차가 평소 습관과 다른 경로로 안내합니다. 기계의 최적화를 믿고 따를 것인가요, 본인의 직관과 계획대로 수정할 것인가요?
A3: 기계의 최적화된 알고리즘을 믿고 따르겠습니다. 자율주행 시스템은 수많은 데이터를 기반으로 가장 안전하고 빠른 경로를 계산한 결과이기 때문에, 개인의 주관적인 직관보다 훨씬 신뢰도가 높다고 판단합니다. 시스템의 매뉴얼에 따라 목적지까지 안정적으로 이동하는 것을 선호합니다.
당신의 MBTI는 ISFJ입니다.
I(내향형): 58.0%
S(감각형): 81.0%
F(감정형): 67.0%
J(판단형): 98.0%


In [23]:
from transformers import pipeline
import torch

def reset_model():
    device = 0 if torch.cuda.is_available() else -1
    return pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7', device=device)

In [21]:
def set_up():
    mbti_labels = {
            '자기 외부에 주의 집중, 외향적인 활동과 사람을 선호': 'E',
            '자기 내부에 주의 집중, 내향적인 휴식과 혼자만의 시간을 선호': 'I',
            '현실주의적, 오감 및 경험에 의존': 'S',
            '이상주의적, 직관 및 영감에 의존': 'N',
            '논리적이고 객관적인 판단을 중시': 'T', 
            '인간관계와 감정적 공감을 중시': 'F',
            '계획적이고 체계적인 생활을 선호': 'J', 
            '즉흥적이고 유연한 상황 대처를 선호': 'P'
        }

    personality_labels = {
            'EI': ['자기 외부에 주의 집중, 외향적인 활동과 사람을 선호', '자기 내부에 주의 집중, 내향적인 휴식과 혼자만의 시간을 선호'],
            'SN': ['현실주의적, 오감 및 경험에 의존', '이상주의적, 직관 및 영감에 의존'],
            'TF': ['논리적이고 객관적인 판단을 중시', '인간관계와 감정적 공감을 중시'],
            'JP': ['계획적이고 체계적인 생활을 선호', '즉흥적이고 유연한 상황 대처를 선호']
        }
    return mbti_labels, personality_labels

In [17]:
import json

def load_questions(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [13]:
def analyze_mbti(clf, answer, targets, m_labels, p_labels, total_score):
    # 답변 분석
    for target in targets:
        candidate_labels = p_labels[target]
        output = clf(
            answer,
            candidate_labels=candidate_labels,
            hypothesis_template='이 사람은 {}하는 사람이다.'
        )        
        for label, score in zip(output['labels'], output['scores']):
            short_label = m_labels[label]
            total_score[short_label] += score
            
    return total_score

In [14]:
def print_result(total_score):
    # 결과 출력
    result = ''
    result += 'E' if total_score['E'] > total_score['I'] else 'I'
    result += 'S' if total_score['S'] > total_score['N'] else 'N'
    result += 'T' if total_score['T'] > total_score['F'] else 'F'
    result += 'J' if total_score['J'] > total_score['P'] else 'P'
    print(f'\n당신의 MBTI는 {result}입니다.')
    
    mbti_detail = {
        'E': {'score': 0, 'description': '외향형'},
        'I': {'score': 0, 'description': '내향형'},
        'S': {'score': 0, 'description': '감각형'},
        'N': {'score': 0, 'description': '직관형'},
        'T': {'score': 0, 'description': '사고형'},
        'F': {'score': 0, 'description': '감정형'},
        'J': {'score': 0, 'description': '판단형'},
        'P': {'score': 0, 'description': '인지형'},                            
    }
    
    sections = [('E', 'I'), ('S', 'N'), ('T', 'F'), ('J', 'P')]
    
    for former, latter in sections:
        mbti_detail[former]['score'] = round(total_score[former] / (total_score[former] + total_score[latter]), 2) * 100
        mbti_detail[latter]['score'] = 100 - mbti_detail[former]['score']
        if mbti_detail[former]['score'] > mbti_detail[latter]['score']:
            print(f'{former}({mbti_detail[former]['description']}): {mbti_detail[former]['score']}%')
        else:
            print(f'{latter}({mbti_detail[latter]['description']}): {mbti_detail[latter]['score']}%')
    
    return result, mbti_detail

In [20]:
import random

def run_process():
    clf = reset_model()
    mbti_labels, personality_labels = set_up()
    questions = load_questions('./src/questions.json')
    total_score = {'E': 0, 'I': 0, 'S': 0, 'N': 0, 'T': 0, 'F': 0, 'J': 0, 'P': 0}

    selected_questions = random.sample(questions, 3)

    print('간이 MBTae 분석을 시작합니다.\n만약 나라면 어떤 생각을 할까에 대하여 답변해주세요.\n(최소 3문장 이상의 답변으로 부탁드립니다!)')

    for i, item in enumerate(selected_questions):
        print(f'\nQ{i+1}: {item['q']}')
        answer = input('답변: ')
        print(f'A{i+1}: {answer}')
        total_score = analyze_mbti(clf, answer, item['targets'], mbti_labels, personality_labels, total_score)
    
    final_mbti, mbti_detail = print_result(total_score)

In [22]:
run_process()

Device set to use cuda:0


간이 MBTae 분석을 시작합니다.
만약 나라면 어떤 생각을 할까에 대하여 답변해주세요.
(최소 3문장 이상의 답변으로 부탁드립니다!)

Q1: 자율 주행 자동차가 평소 습관과 다른 경로로 안내합니다. 기계의 최적화를 믿고 따를 것인가요, 본인의 직관과 계획대로 수정할 것인가요?
A1: 기계의 최적화된 알고리즘을 믿고 따르겠습니다. 자율주행 시스템은 수많은 데이터를 기반으로 가장 안전하고 빠른 경로를 계산한 결과이기 때문에, 개인의 주관적인 직관보다 훨씬 신뢰도가 높다고 판단합니다. 시스템의 매뉴얼에 따라 목적지까지 안정적으로 이동하는 것을 선호합니다.

Q2: 새로운 스포츠 경기 규칙을 만든다면, 기존의 데이터와 물리적 한계를 중시하시겠습니까, 아니면 창의적인 재미를 중시하시겠습니까?
A2: 과거 10년간의 경기 데이터를 기반으로 가장 공정한 규칙을 만들겠습니다. 선수의 신체 조건과 물리적 한계를 수치화하여 판정 시 논란이 없도록 명확한 가이드라인을 세우는 데 집중할 것입니다. 재미보다는 공정성과 운영의 안정성이 제 1원칙입니다.

Q3: 관중 앞에서 전공 지식을 발표할 때, 청중의 반응에 에너지를 얻나요? 자료 준비 시 통계 수치와 미래 비전 중 무엇을 더 강조할까요?
A3: 청중의 반응보다는 제가 준비한 자료의 정확도에 더 집중합니다. 발표 시 미래의 막연한 비전보다는 현재 검증된 통계 수치와 논문 자료를 근거로 제시할 것입니다. 그래야만 청중에게 가장 신뢰도 높은 정보를 전달할 수 있다고 믿기 때문입니다.

당신의 MBTI는 ESTJ입니다.
E(외향형): 69.0%
S(감각형): 92.0%
T(사고형): 82.0%
J(판단형): 96.0%
